### SEMMA · Fase 3: Modify

Limpieza y transformación de los datos. Las reglas se definen con el grupo de entrenamiento y se aplican igual al de prueba.

- **Entradas:** `semma/data/interim/train.csv`, `semma/data/interim/test.csv`
- **Salidas:**
  - `semma/data/interim/train_clean.csv`, `semma/data/interim/test_clean.csv`: datos limpios, sin escalar
  - `semma/data/processed/train_prepared.csv`, `semma/data/processed/test_prepared.csv`: datos transformados
  - `semma/models/preprocessor.joblib`: preprocesador ajustado

#### Librerías

In [1]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

#### Rutas

Relativas a la carpeta del notebook.

In [2]:
INTERIM_DIR = Path("../data/interim")
PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

#### Carga de datos

In [3]:
train = pd.read_csv(INTERIM_DIR / "train.csv")
test = pd.read_csv(INTERIM_DIR / "test.csv")

print(f"Entrenamiento: {train.shape}")
print(f"Prueba:        {test.shape}")

Entrenamiento: (275, 8)
Prueba:        (69, 8)


#### Definición de variables

In [4]:
TARGET = "species"
ID = "row_id"
NUMERIC = ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]
CATEGORICAL = ["island", "sex"]
FEATURES = NUMERIC + CATEGORICAL

#### Eliminación de registros sin medidas

In [5]:
def drop_without_measures(df):
    mask = df[NUMERIC].isna().all(axis=1)
    return df.loc[~mask].copy(), df.loc[mask, ID].tolist()

train, dropped_train = drop_without_measures(train)
test, dropped_test = drop_without_measures(test)

print(f"Eliminados en entrenamiento: {dropped_train}")
print(f"Eliminados en prueba:        {dropped_test}")
print(f"Entrenamiento: {len(train)} | Prueba: {len(test)}")

Eliminados en entrenamiento: [3, 339]
Eliminados en prueba:        []
Entrenamiento: 273 | Prueba: 69


#### Tratamiento de `sex`

Todo valor distinto de `MALE` o `FEMALE` pasa a `UNKNOWN`.

In [6]:
VALID_SEX = ["MALE", "FEMALE"]

def clean_sex(df):
    df = df.copy()
    df["sex"] = df["sex"].where(df["sex"].isin(VALID_SEX), "UNKNOWN")
    return df

train = clean_sex(train)
test = clean_sex(test)

pd.DataFrame({
    "train": train["sex"].value_counts(),
    "test": test["sex"].value_counts(),
}).fillna(0).astype(int)

,train,test
sex,,
FEMALE,134,31
MALE,133,35
UNKNOWN,6,3


#### Verificación de datos limpios

In [7]:
pd.DataFrame({
    "train": train[FEATURES + [TARGET]].isna().sum(),
    "test": test[FEATURES + [TARGET]].isna().sum(),
})

,train,test
culmen_length_mm,0,0
culmen_depth_mm,0,0
flipper_length_mm,0,0
body_mass_g,0,0
island,0,0
sex,0,0
species,0,0


#### Guardado de datos limpios

In [8]:
train.to_csv(INTERIM_DIR / "train_clean.csv", index=False)
test.to_csv(INTERIM_DIR / "test_clean.csv", index=False)

print(f"Guardado: {INTERIM_DIR / 'train_clean.csv'}")
print(f"Guardado: {INTERIM_DIR / 'test_clean.csv'}")

Guardado: ../data/interim/train_clean.csv
Guardado: ../data/interim/test_clean.csv


#### Preprocesador

- Numéricas: imputación por mediana y estandarización.
- Categóricas: codificación one-hot; una categoría no vista se codifica como ceros.

In [9]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    [
        ("num", numeric_pipe, NUMERIC),
        ("cat", categorical_pipe, CATEGORICAL),
    ],
    verbose_feature_names_out=False,
)

#### Ajuste y transformación

El preprocesador se ajusta solo con el grupo de entrenamiento.

In [10]:
preprocessor.fit(train[FEATURES])

feature_names = preprocessor.get_feature_names_out()

def transform(df):
    X = pd.DataFrame(preprocessor.transform(df[FEATURES]), columns=feature_names, index=df.index)
    return pd.concat([df[[ID]], X, df[[TARGET]]], axis=1)

train_prepared = transform(train)
test_prepared = transform(test)

print(f"Variables resultantes ({len(feature_names)}): {list(feature_names)}")
train_prepared.head()

Variables resultantes (10): ['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g', 'island_Biscoe', 'island_Dream', 'island_Torgersen', 'sex_FEMALE', 'sex_MALE', 'sex_UNKNOWN']


,row_id,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,island_Biscoe,island_Dream,island_Torgersen,sex_FEMALE,sex_MALE,sex_UNKNOWN,species
0,193,0.440701,0.176677,-0.964733,-0.663196,0.0,1.0,0.0,1.0,0.0,0.0,Chinstrap
1,98,-1.970312,-0.528187,-1.605714,-1.587406,0.0,1.0,0.0,1.0,0.0,0.0,Adelie
2,155,0.293464,0.780846,-0.893513,-0.817231,0.0,1.0,0.0,1.0,0.0,0.0,Chinstrap
3,40,-1.344553,0.428414,-1.320833,-1.279336,0.0,1.0,0.0,1.0,0.0,0.0,Adelie
4,281,0.440701,-1.132355,1.456751,1.370064,1.0,0.0,0.0,0.0,1.0,0.0,Gentoo


#### Verificación de la transformación

In [11]:
pd.DataFrame({
    "train_media": train_prepared[NUMERIC].mean(),
    "train_desv": train_prepared[NUMERIC].std(ddof=0),
    "test_media": test_prepared[NUMERIC].mean(),
    "test_desv": test_prepared[NUMERIC].std(ddof=0),
}).round(3)

,train_media,train_desv,test_media,test_desv
culmen_length_mm,0.0,1.0,0.106,1.012
culmen_depth_mm,-0.0,1.0,0.005,0.964
flipper_length_mm,0.0,1.0,0.130,0.993
body_mass_g,-0.0,1.0,0.083,0.930


#### Parámetros aprendidos

In [12]:
scaler = preprocessor.named_transformers_["num"].named_steps["scaler"]
display(pd.DataFrame({"media": scaler.mean_, "desviacion": scaler.scale_}, index=NUMERIC).round(2))

encoder = preprocessor.named_transformers_["cat"].named_steps["onehot"]
for col, cats in zip(CATEGORICAL, encoder.categories_):
    print(f"{col}: {list(cats)}")

,media,desviacion
culmen_length_mm,43.81,5.43
culmen_depth_mm,17.15,1.99
flipper_length_mm,200.55,14.04
body_mass_g,4188.19,811.50


island: ['Biscoe', 'Dream', 'Torgersen']
sex: ['FEMALE', 'MALE', 'UNKNOWN']


#### Guardado de resultados

In [13]:
train_prepared.to_csv(PROCESSED_DIR / "train_prepared.csv", index=False)
test_prepared.to_csv(PROCESSED_DIR / "test_prepared.csv", index=False)
joblib.dump(preprocessor, MODELS_DIR / "preprocessor.joblib")

for path in [
    PROCESSED_DIR / "train_prepared.csv",
    PROCESSED_DIR / "test_prepared.csv",
    MODELS_DIR / "preprocessor.joblib",
]:
    print(f"Guardado: {path}")

Guardado: ../data/processed/train_prepared.csv
Guardado: ../data/processed/test_prepared.csv
Guardado: ../models/preprocessor.joblib
